# 02. Comfort Pick Classification

**Goal:** establish an operational definition to label every player-game as:
- `is_comfort = True` (comfort champion)
- `is_comfort = False` (off-comfort / assigned champion)

**Criteria:**
1. Minimum sample of $\ge 8$ games per player in the split.
2. **Comfort:** $\ge 4$ games on the champion within the split.
3. **No-comfort:** $\le 2$ games on the champion within the split.
*(Games with exactly 3 picks are dropped to avoid an ambiguous middle zone.)*

**A note on causality:** this definition is **retrospective**, not prospective.
It considers the total champion volume across the entire split rather than strictly
prior games. This represents an explicit methodological constraint: the model captures
whether a champion was a persistent comfort pick throughout the season, not whether
the coaching staff held proven historical safety prior to game day.

In [ ]:
import pandas as pd
import numpy as np

# Load the cleaned dataset from step 01
DATA_PATH = "data/processed/player_games_clean.csv"
df = pd.read_csv(DATA_PATH)

print(f"Total records loaded: {len(df)}")
df.head(3)

In [ ]:
# Minimum number of games a player must have in a split to be analyzable at all
MIN_GAMES_PLAYER = 8

# Total game volume for that player in that split/year
df["player_total_games"] = df.groupby(["playername", "year", "split"])["gameid"].transform("nunique")

# Filter
df_filtered = df[df["player_total_games"] >= MIN_GAMES_PLAYER].copy()
print(f"Rows after filtering players with < {MIN_GAMES_PLAYER} games: {len(df_filtered)}")

In [ ]:
# 1. Count how many times that champion was played in that split
df_filtered["champ_games_split"] = df_filtered.groupby(
    ["playername", "year", "split", "champion"]
)["gameid"].transform("count")

# 2. Assign labels
conditions = [
    df_filtered["champ_games_split"] >= 4,
    df_filtered["champ_games_split"] <= 2,
]
choices = [True, False]

df_filtered["is_comfort"] = np.select(conditions, choices, default=np.nan)

# 3. Drop the neutral/ambiguous zone (exactly 3 games)
df_classified = df_filtered.dropna(subset=["is_comfort"]).copy()
df_classified["is_comfort"] = df_classified["is_comfort"].astype(bool)

In [ ]:
# Class-balance summary by role.
# `.reindex(columns=[False, True])` makes this robust even if some role has
# zero games in one of the two classes in a smaller/filtered dataset -- the
# original `.unstack()` call would otherwise silently drop that column and
# break the fixed `summary.columns = [...]` assignment below it.
summary = (
    df_classified.groupby(["position", "is_comfort"]).size()
    .unstack(fill_value=0)
    .reindex(columns=[False, True], fill_value=0)
)
summary.columns = ["No Comfort (<=2)", "Comfort (>=4)"]
summary["% Comfort"] = (summary["Comfort (>=4)"] / summary.sum(axis=1)) * 100
summary.round(1)

In [ ]:
OUT_PATH = "data/processed/player_games_classified.csv"
df_classified.to_csv(OUT_PATH, index=False)
print(f"Classified dataset saved to {OUT_PATH} with {len(df_classified)} rows.")

## Optional / not run by default — prospective comfort classification

The cell below is a **starting point**, not part of the main pipeline. It
recomputes `is_comfort` using only games played *before* the current one
(a running count, ordered by date), which is closer to "did the coaching
staff already have evidence this was safe" than the retrospective
definition used above. It's left here, clearly separated, so you can
compare the two definitions side by side before deciding whether to make
the prospective version the default for `03_stats_analysis.ipynb`.

Note the trade-off: a prospective definition throws away the first few
games on every champion (there's no "prior evidence" yet), which will
shrink your comfort-labeled sample further on top of the `MIN_GAMES_PLAYER`
filter already applied above.

In [ ]:
RUN_PROSPECTIVE_VERSION = False  # flip to True to explore this alternative

if RUN_PROSPECTIVE_VERSION:
    df_sorted = df_filtered.sort_values(["playername", "year", "split", "date"])
    df_sorted["champ_games_before_this"] = df_sorted.groupby(
        ["playername", "year", "split", "champion"]
    ).cumcount()
    # cumcount() is 0-indexed (0 = first time playing this champion in the split),
    # so ">= 4" here means "this is at least the 5th time" -- adjust the
    # threshold to taste, it does not have to match the retrospective one.
    df_sorted["is_comfort_prospective"] = df_sorted["champ_games_before_this"] >= 4

    prospective_rate = df_sorted["is_comfort_prospective"].mean()
    print(f"Share of games labeled comfort under the prospective definition: {prospective_rate:.1%}")
    print("Compare this to the retrospective is_comfort rate above before choosing one for 03.")